In [1]:
# Preparing pathing
%load_ext autoreload
%autoreload 2
from titanic_ml import paths
import matplotlib.pyplot as plt
import pandas as pd
from titanic_ml.common.data.eda import summarize_categorical_column, summarize_numerical_column
from titanic_ml.common.data.eda import run_eda 


from ast import For
from cmath import exp

from titanic_ml.common.experiments.runner import run_experiments, run_experiment_group_workflow
from titanic_ml.common.experiments.config import ALL_EXPERIMENTS, Final_experiments_baseline, Final_experiments_v1
from titanic_ml.common.experiments.report import experiment_report, experiment_group_summary_report, baseline_summary_to_markdown, workflow_report
from titanic_ml.common.experiments.save_load import save_results, load_results, save_configs, load_configs, save_feature_effects, load_feature_effects
from titanic_ml.common.experiments.compare import leaderboard, compare_experiment_groups, summarize_group_comparison, titanic_notes_leaderboard, model_progression, analyze_feature_effect, domain_best_by_model_with_baseline_delta, domain_best_by_model, recommended_by_domain_for_model
from titanic_ml.common.models.registry import MODEL_REGISTRY
from titanic_ml.feature_engineering import add_family_features, add_has_cabin, add_title, add_full_title_feature
from titanic_ml.common.data.eda import sample_dataframe


In [52]:
TARGET = "Survived"
ALL_EXPERIMENTS = ALL_EXPERIMENTS
for experiment_name, exp_config in ALL_EXPERIMENTS.items():
    print(f"Experiment: {experiment_name}")

train_df = pd.read_csv(paths.TRAIN_PATH)

exp_configs = ALL_EXPERIMENTS["fe11__age_bin"]

# # Line to rerun all experiments to update the results with the latest code changes. This will take a while.
# # Uncomment to run all experiments and update results.

# for Name, exp_config in ALL_EXPERIMENTS.items():
#     print(f"Running {Name} experiments...")
#     exp_result = run_experiments(train_df, exp_config, target=TARGET, verbose=True, debug=True)
#     result_df = save_results(exp_result)
#     save_configs(exp_config)
#     if Name != 'baseline__raw':
#         comparison = compare_experiment_groups(
#             results_df=result_df,
#             reference_group="baseline__raw",
#             compare_groups=[Name],
#         )
#         feature_effect = analyze_feature_effect(comparison)
#         save_feature_effects(feature_effect)



Experiment: baseline__raw
Experiment: fe01__family
Experiment: fe02__has_cabin
Experiment: fe03__deck
Experiment: fe04__cabin_features
Experiment: fe05__title
Experiment: fe06__age_imputation_title
Experiment: fe07__age_imputation_title_pclass
Experiment: fe08__fare_per_family_member
Experiment: fe09__ticket_group_size
Experiment: fe10__fare_per_ticket_member
Experiment: fe11__age_bin
Experiment: fe12__sex_pclass
Experiment: cb01__age_and_bins
Experiment: cb02__age_imputed_title_and_bins
Experiment: cb03__age_imputed_title_Pclass_and_bins
Experiment: cb04__fare_and_fare_per_family
Experiment: cb05__fare_and_fare_per_ticket
Experiment: cb06__all_fare_features
Experiment: cb07__family_features
Experiment: cb08__pclass_sex_features
Experiment: ab01__age_and_bins_without_fare
Experiment: ab02__age_imputed_title_and_bins_without_fare
Experiment: ab03__age_imputed_title_Pclass_and_bins_without_fare


In [3]:
# print("Experiment Configurations:")
# print(exp_configs)
# for exp_config in exp_configs:
#     print(exp_config)

In [4]:
# Work flow for running an experiment group, comparing it to the baseline, and generating a report. 
# This is the main workflow for analyzing the results of an experiment group and generating insights from it.
workflow = run_experiment_group_workflow(
    df=train_df,
    experiment_configs=exp_configs,
    target=TARGET,
    save=True,
)
# print("Workflow completed. Here are the results:")
# print("Comparison between baseline and feature engineering group:")
# print(workflow["comparison"])
# print("Summary of comparison:")
# print(workflow["summary"])
# print("Leaderboard:")
# print(workflow["leaderboard"])

In [5]:
# Generate a full report for the workflow, including the comparison, summary, and leaderboard. 
# This will be a markdown report that can be easily shared and visualized.
# Mainly used for generating the report for the notebook, but can also be used for generating reports for individual experiment groups or comparisons.
full_report = workflow_report(workflow, top_n=20)
print("Full workflow report:")
print()
print('Report')
print(full_report['report'])
print()
print('Leaderboard')
print(full_report['leaderboard'])
print()
# For combos:
all_results = load_results()
references = ['fe12__sex_pclass']
for reference in references:
    comparison = compare_experiment_groups(
                results_df=all_results,
                reference_group=reference,
                compare_groups=[exp_configs],
            )
    print(f"Comparison summary:")
    print(comparison[["reference_group", "compare_group", "model_name", "test_accuracy_mean_delta", "test_f1_mean_delta"]].to_markdown())
    print()


Full workflow report:

Report
### fe11__age_bin

_Description pending._

<details>
<summary>Conclusion</summary>


#### Interpretation

- Verdict: mixed
- Recommended for specific models:
  - decision_tree: test_accuracy_mean: 0.006
    - Secondary gains:
      - test_f1_mean: 0.02
  - random_forest: test_accuracy_mean: 0.011
    - Secondary gains:
      - test_f1_mean: 0.015
  - extra_trees: test_accuracy_mean: 0.015
    - Secondary gains:
      - test_f1_mean: 0.016


#### Conclusion

_Conclusion pending._

</details>

<details>
<summary>Experiment details</summary>

#### Comparison vs baseline__raw

| reference_group   | compare_group   | model_name    |   test_accuracy_mean_reference |   test_accuracy_mean_compare |   test_accuracy_mean_delta |   test_f1_mean_reference |   test_f1_mean_compare |   test_f1_mean_delta |
|:------------------|:----------------|:--------------|-------------------------------:|-----------------------------:|---------------------------:|------------------

In [6]:
# Leaderboard without ablations
current_leaderboard = titanic_notes_leaderboard(all_results, top_n=10, selection="exclude")
print("Current leaderboard:")
print(current_leaderboard)

Current leaderboard:
| experiment                                   | model_name    |   test_accuracy_mean |   test_f1_mean |
|:---------------------------------------------|:--------------|---------------------:|---------------:|
| fe05__title__xgb                             | xgb           |                0.836 |          0.772 |
| fe05__title__svc                             | svc           |                0.834 |          0.771 |
| fe11__age_bin__random_forest                 | random_forest |                0.833 |          0.759 |
| fe09__ticket_group_size__svc                 | svc           |                0.832 |          0.77  |
| fe05__title__random_forest                   | random_forest |                0.832 |          0.768 |
| fe04__cabin_features__xgb                    | xgb           |                0.832 |          0.767 |
| cb02__age_imputed_title_and_bins__svc        | svc           |                0.831 |          0.767 |
| cb03__age_imputed_title_Pclass_a

In [7]:
thresholds = {
    "accuracy": 0.003,
    "f1": -0.01,
}

for model in MODEL_REGISTRY:

    result = recommended_by_domain_for_model(
        results_df=all_results,
        model_name=model,
        thresholds=thresholds,
    )

    print(f"\nBest candidates for {model}")
    print("=" * 50)

    print("\nRecommended:")
    for recommendation in result["recommended"]:
        print(
            recommendation["domain"],
            "->",
            recommendation["group"],
            recommendation["deltas"],
        )
    print()
    print("recommended list:")
    for recommendation in result["recommended"]:
        print(recommendation["group"], end=", ")
    print()
    print("\nFull domain results:")
    print(result["df"].to_markdown(index=False))


Best candidates for logreg

Recommended:
title -> fe05__title {'accuracy': 0.039, 'f1': 0.052}
age -> cb03__age_imputed_title_Pclass_and_bins {'accuracy': 0.02, 'f1': 0.022}
ablation -> ab03__age_imputed_title_Pclass_and_bins_without_fare {'accuracy': 0.016, 'f1': 0.017}
pclass_sex -> cb08__pclass_sex_features {'accuracy': 0.016, 'f1': 0.0}
family -> fe01__family {'accuracy': 0.009, 'f1': 0.008}
cabin -> fe04__cabin_features {'accuracy': 0.005, 'f1': 0.011}
fare -> fe08__fare_per_family_member {'accuracy': 0.003, 'f1': 0.004}

recommended list:
fe05__title, cb03__age_imputed_title_Pclass_and_bins, ab03__age_imputed_title_Pclass_and_bins_without_fare, cb08__pclass_sex_features, fe01__family, fe04__cabin_features, fe08__fare_per_family_member, 

Full domain results:
| domain     | group                                                | experiment                                                   | model_name   |   test_accuracy_mean |   test_f1_mean |   accuracy_delta_vs_baseline |   f1_

In [8]:
for model in Final_experiments_baseline.keys():
    print(model)

logreg
knn
svc
decision_tree
random_forest
extra_trees
xgb


In [9]:
from titanic_ml.feature_engineering import age_bin_transformer, TitleTransformer
from titanic_ml.feature_engineering import add_age_bin, add_full_title_feature


# old_df = add_age_bin(train_df)
old_df = add_full_title_feature(train_df)

transformer = TitleTransformer()
new_df = transformer.fit_transform(train_df)

print("Old DataFrame with Title:")
print(old_df["Title"].head())
print("\nNew DataFrame with Title:")
print(new_df["Title"].head())

comparison = pd.DataFrame({
    "old": old_df["Title"],
    "new": new_df["Title"],
})

print(
    comparison[
        comparison["old"] != comparison["new"]
    ]
)

pd.testing.assert_series_equal(
    old_df["Title"],
    new_df["Title"],
    check_names=False,
)



Old DataFrame with Title:
0      Mr
1     Mrs
2    Miss
3     Mrs
4      Mr
Name: Title, dtype: object

New DataFrame with Title:
0      Mr
1     Mrs
2    Miss
3     Mrs
4      Mr
Name: Title, dtype: object
Empty DataFrame
Columns: [old, new]
Index: []


In [10]:
from titanic_ml.feature_engineering.age_impute import age_imputed_by_title
from titanic_ml.feature_engineering.sklearn_compatible.Age_Imputer import age_imputer_title
from sklearn.pipeline import Pipeline

old_df = add_full_title_feature(train_df)
old_df = age_imputed_by_title(old_df)

new_pipeline = Pipeline([
    ("title", TitleTransformer()),
    (
        "age_imputer",
        age_imputer_title()
    ),
])

new_df = new_pipeline.fit_transform(train_df)

In [11]:
pd.testing.assert_series_equal(
    old_df["Age"],
    new_df["Age"],
    check_names=False,
)

In [12]:
train_test = pd.DataFrame({
    "Title": [
        "Mr",
        "Mr",
        "Mrs",
        "Mrs",
    ],
    "Age": [
        20,
        40,
        30,
        50,
    ],
})

validation_test = pd.DataFrame({
    "Title": [
        "Mr",
        "Mrs",
        "Miss",
    ],
    "Age": [
        None,
        None,
        None,
    ],
})

In [13]:
imputer = age_imputer_title()

imputer.fit(train_test)

result = imputer.transform(validation_test)
print(result)

  Title   Age
0    Mr  30.0
1   Mrs  40.0
2  Miss  35.0


C:\Users\jonma\Documents\GitHub\Machine_learning\Titanic\titanic_ml\sklearn_compatible\GroupedImputer.py:121: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(self.global_value_)


In [14]:
from sklearn.base import clone

transformer = age_imputer_title()

clone(transformer)

,target_col,'Age'
,group_cols,['Title']
,agg_func,'median'
,fallback_to_global,True


In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from titanic_ml.common.models.registry import MODEL_REGISTRY
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler, RobustScaler
from titanic_ml.common.experiments.save_load import save_results, load_results, save_configs, load_configs, save_feature_effects, load_feature_effects
from titanic_ml.common.experiments.compare import compare_experiment_groups, summarize_group_comparison, leaderboard, analyze_feature_effect


def build_preprocessor(preprocessing_config):
    numeric_features = preprocessing_config.get("numeric_features", [])
    onehot_features = preprocessing_config.get("onehot_features", [])
    ordinal_features = preprocessing_config.get("ordinal_features", [])

    numeric_imputer = preprocessing_config.get("numeric_imputer", "median")
    categorical_imputer = preprocessing_config.get("categorical_imputer", "most_frequent")
    scaler = preprocessing_config.get("scaler", None)

    transformers = []

    if numeric_features:
        numeric_steps = []

        if numeric_imputer:
            numeric_steps.append(
                ("imputer", SimpleImputer(strategy=numeric_imputer))
            )

        if scaler == "standard":
            numeric_steps.append(("scaler", StandardScaler()))
        elif scaler == "minmax":
            numeric_steps.append(("scaler", MinMaxScaler()))
        elif scaler == "robust":
            numeric_steps.append(("scaler", RobustScaler()))
        elif scaler is not None:
            raise ValueError(f"Unsupported scaler: {scaler}")

        numeric_pipeline = Pipeline(numeric_steps)

        transformers.append(
            ("num", numeric_pipeline, numeric_features)
        )

    if onehot_features:
        onehot_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy=categorical_imputer)),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ])

        transformers.append(
            ("onehot", onehot_pipeline, onehot_features)
        )

    if ordinal_features:
        ordinal_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy=categorical_imputer)),
            ("encoder", OrdinalEncoder()),
        ])

        transformers.append(
            ("ordinal", ordinal_pipeline, ordinal_features)
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=True,
    )

In [16]:
from titanic_ml.feature_engineering.age_impute import age_imputed_by_title

Pipeline([
    ("title", TitleTransformer()),
    ("age_imputer", age_imputer_title()),
])

old_df = age_imputed_by_title(train_df)
new_df = Pipeline([
    ("title", TitleTransformer()),
    ("age_imputer", age_imputer_title()),
]).fit_transform(train_df)

pd.testing.assert_series_equal(
    old_df["Age"],
    new_df["Age"],
    check_names=False,
)

In [17]:
train_test_df = pd.DataFrame({
    "Name": [
        "Smith, Mr. John",
        "Brown, Mr. James",
        "Jones, Mrs. Alice",
        "White, Mrs. Mary",
    ],
    "Age": [
        20,
        40,
        30,
        50,
    ],
})

validation_test_df = pd.DataFrame({
    "Name": [
        "Green, Mr. Peter",
        "Black, Mrs. Anna",
    ],
    "Age": [
        None,
        None,
    ],
})

pipeline = Pipeline([
    ("title", TitleTransformer()),
    ("age_imputer", age_imputer_title()),
])

pipeline.fit(train_test_df)

result = pipeline.transform(validation_test_df)

print(result[["Name", "Title", "Age"]])

               Name Title   Age
0  Green, Mr. Peter    Mr  30.0
1  Black, Mrs. Anna   Mrs  40.0


C:\Users\jonma\Documents\GitHub\Machine_learning\Titanic\titanic_ml\sklearn_compatible\GroupedImputer.py:121: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(self.global_value_)


In [18]:
train_test_df = pd.DataFrame({
    "Name": [
        "Smith, Mr. John",
        "Brown, Mr. James",
        "Jones, Mrs. Alice",
        "White, Mrs. Mary",
    ],
    "Age": [
        20,
        40,
        30,
        50,
    ],
})

validation_test_df = pd.DataFrame({
    "Name": [
        "Green, Dona. Jane",
    ],
    "Age": [
        None,
    ],
})

pipeline = Pipeline([
    ("title", TitleTransformer()),
    ("age_imputer", age_imputer_title()),
])

pipeline.fit(train_test_df)

result = pipeline.transform(validation_test_df)

print(result[["Title", "Age"]])

assert result.loc[0, "Age"] == 35

  Title   Age
0  Dona  35.0


C:\Users\jonma\Documents\GitHub\Machine_learning\Titanic\titanic_ml\sklearn_compatible\GroupedImputer.py:121: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(self.global_value_)


In [19]:
train_test_df = pd.DataFrame({
    "Name": [
        "Smith, Mr. John",
        "Brown, Mr. James",
    ],
    "Age": [
        20,
        40,
    ],
})

validation_test_df = pd.DataFrame({
    "Name": [
        "Green, Mr. Peter",
        "Black, Mr. Paul",
    ],
    "Age": [
        None,
        100,
    ],
})

pipeline = Pipeline([
    ("title", TitleTransformer()),
    ("age_imputer", age_imputer_title()),
])

pipeline.fit(train_test_df)

result = pipeline.transform(validation_test_df)

print(result[["Title", "Age"]])

assert result.loc[0, "Age"] == 30

  Title    Age
0    Mr   30.0
1    Mr  100.0


In [20]:
from sklearn.base import clone

pipeline = Pipeline([
    ("title", TitleTransformer()),
    ("age_imputer", age_imputer_title()),
])

cloned_pipeline = clone(pipeline)

result = cloned_pipeline.fit_transform(train_test_df)

print(result.head())

               Name  Age Title
0   Smith, Mr. John   20    Mr
1  Brown, Mr. James   40    Mr


In [21]:
pipeline.fit(train_test_df)

imputer = pipeline.named_steps["age_imputer"]

print(imputer.group_values_)
print(imputer.global_value_)


Title
Mr    30.0
Name: Age, dtype: float64
30.0


In [22]:

from titanic_ml.feature_engineering.sklearn_compatible.Age_Imputer import age_imputer_title_pclass


cb03_pipeline = Pipeline([
    ("title", TitleTransformer()),
    ("age_imputer", age_imputer_title_pclass()),
    ("age_bin", age_bin_transformer()),
])

new_df = cb03_pipeline.fit_transform(train_df)

In [23]:

from titanic_ml.feature_engineering.age_impute import age_imputed_by_title_pclass


old_df = age_imputed_by_title_pclass(train_df)
old_df = add_age_bin(old_df)

pd.testing.assert_series_equal(
    old_df["Age"],
    new_df["Age"],
    check_names=False,
)

pd.testing.assert_series_equal(
    old_df["Age_bin"],
    new_df["Age_bin"],
    check_names=False,
)

In [24]:
from titanic_ml.common.experiments.v0_revised_runner import run_experiments as run_test
from titanic_ml.common.experiments.v0_config import fe11_config

print(fe11_config)
print()
for key in fe11_config:
    print(key)
print()
for key, value in fe11_config['logreg__age_bin'].items():
    print(f"{key}: {value}")

{'logreg__age_bin': {'name': 'fe11__age_bin__logreg', 'features': ['SibSp', 'Parch', 'Sex', 'Embarked', 'Pclass', 'Age_bin'], 'preprocessing': {'numeric_features': ['SibSp', 'Parch'], 'onehot_features': ['Sex', 'Embarked'], 'ordinal_features': ['Pclass', 'Age_bin'], 'numeric_imputer': 'median', 'categorical_imputer': 'most_frequent', 'scaler': 'standard'}, 'model_name': 'logreg', 'model_params': {'max_iter': 1000, 'random_state': 42}, 'evaluation': {'method': 'cross_validation', 'cv': 5, 'scoring': ['accuracy', 'precision', 'recall', 'f1'], 'return_train_score': True, 'n_jobs': -1}, 'notes': 'Feature engineering 11: replaces Age with Age_bin.', 'stage': 'fe11', 'feature_group': 'age_bin', 'group': 'fe11__age_bin', 'feature_pipeline': [('age_bin', BinTransformer(bins=[0, 14, 35, 60, 100], labels=['0', '2', '3', '1'],
               output_col='Age_bin', source_col='Age'))], 'domain': 'age'}, 'knn__age_bin': {'name': 'fe11__age_bin__knn', 'features': ['SibSp', 'Parch', 'Sex', 'Embarked',

In [25]:
print('Running old experiments...')
old_exp_result = run_experiments(train_df, exp_configs, target=TARGET, verbose=True, debug=True)


Running old experiments...
Running experiment: fe11__age_bin__logreg
Running config: {'name': 'fe11__age_bin__logreg', 'features': ['SibSp', 'Parch', 'Sex', 'Embarked', 'Pclass', 'Age_bin'], 'feature_engineering': [<function add_age_bin at 0x00000223561115A0>], 'preprocessing': {'numeric_features': ['SibSp', 'Parch'], 'onehot_features': ['Sex', 'Embarked'], 'ordinal_features': ['Pclass', 'Age_bin'], 'numeric_imputer': 'median', 'categorical_imputer': 'most_frequent', 'scaler': 'standard'}, 'model_name': 'logreg', 'model_params': {'max_iter': 1000, 'random_state': 42}, 'evaluation': {'method': 'cross_validation', 'cv': 5, 'scoring': ['accuracy', 'precision', 'recall', 'f1'], 'return_train_score': True, 'n_jobs': -1}, 'notes': "Feature engineering 11: Creating age bins, expected to give better results then using raw age values, by better evaluating the survival chances based on the passenger's age group.", 'stage': 'fe11', 'feature_group': 'age_bin', 'group': 'fe11__age_bin', 'domain': '

In [26]:
print('Running new experiments...')
new_exp_result = run_test(train_df, fe11_config, target=TARGET, verbose=True, debug=True)

Running new experiments...
Running experiment: fe11__age_bin__logreg
Running config: {'name': 'fe11__age_bin__logreg', 'features': ['SibSp', 'Parch', 'Sex', 'Embarked', 'Pclass', 'Age_bin'], 'preprocessing': {'numeric_features': ['SibSp', 'Parch'], 'onehot_features': ['Sex', 'Embarked'], 'ordinal_features': ['Pclass', 'Age_bin'], 'numeric_imputer': 'median', 'categorical_imputer': 'most_frequent', 'scaler': 'standard'}, 'model_name': 'logreg', 'model_params': {'max_iter': 1000, 'random_state': 42}, 'evaluation': {'method': 'cross_validation', 'cv': 5, 'scoring': ['accuracy', 'precision', 'recall', 'f1'], 'return_train_score': True, 'n_jobs': -1}, 'notes': 'Feature engineering 11: replaces Age with Age_bin.', 'stage': 'fe11', 'feature_group': 'age_bin', 'group': 'fe11__age_bin', 'feature_pipeline': [('age_bin', BinTransformer(bins=[0, 14, 35, 60, 100], labels=['0', '2', '3', '1'],
               output_col='Age_bin', source_col='Age'))], 'domain': 'age'}
Pipeline steps:
  age_bin: BinTr

In [27]:
metric_columns = [
    "test_accuracy_mean",
    "test_accuracy_std",
    "train_accuracy_mean",
    "train_accuracy_std",

    "test_precision_mean",
    "test_precision_std",
    "train_precision_mean",
    "train_precision_std",

    "test_recall_mean",
    "test_recall_std",
    "train_recall_mean",
    "train_recall_std",

    "test_f1_mean",
    "test_f1_std",
    "train_f1_mean",
    "train_f1_std",
]

result_1 = old_exp_result
result_2 = new_exp_result

print(
    result_1[metric_columns].equals(
        result_2[metric_columns]
    )
)

True


In [28]:
for x in range(10):
    text_exp_result_1 = run_test(train_df, fe11_config, target=TARGET, verbose=False, debug=False)
    text_exp_result_2 = run_test(train_df, fe11_config, target=TARGET, verbose=False, debug=False)
    if not text_exp_result_1.equals(text_exp_result_2):
        print("Inconsistent results found in iteration", x)
    else:
        print("Consistent results found in iteration", x)

Inconsistent results found in iteration 0
Inconsistent results found in iteration 1
Inconsistent results found in iteration 2
Inconsistent results found in iteration 3
Inconsistent results found in iteration 4
Inconsistent results found in iteration 5
Inconsistent results found in iteration 6
Inconsistent results found in iteration 7
Inconsistent results found in iteration 8
Inconsistent results found in iteration 9


In [29]:
text_exp_result_1 = run_test(train_df, fe11_config, target=TARGET, verbose=True, debug=True)
text_exp_result_2 = run_test(train_df, fe11_config, target=TARGET, verbose=True, debug=True)

Running experiment: fe11__age_bin__logreg
Running config: {'name': 'fe11__age_bin__logreg', 'features': ['SibSp', 'Parch', 'Sex', 'Embarked', 'Pclass', 'Age_bin'], 'preprocessing': {'numeric_features': ['SibSp', 'Parch'], 'onehot_features': ['Sex', 'Embarked'], 'ordinal_features': ['Pclass', 'Age_bin'], 'numeric_imputer': 'median', 'categorical_imputer': 'most_frequent', 'scaler': 'standard'}, 'model_name': 'logreg', 'model_params': {'max_iter': 1000, 'random_state': 42}, 'evaluation': {'method': 'cross_validation', 'cv': 5, 'scoring': ['accuracy', 'precision', 'recall', 'f1'], 'return_train_score': True, 'n_jobs': -1}, 'notes': 'Feature engineering 11: replaces Age with Age_bin.', 'stage': 'fe11', 'feature_group': 'age_bin', 'group': 'fe11__age_bin', 'feature_pipeline': [('age_bin', BinTransformer(bins=[0, 14, 35, 60, 100], labels=['0', '2', '3', '1'],
               output_col='Age_bin', source_col='Age'))], 'domain': 'age'}
Pipeline steps:
  age_bin: BinTransformer
  preprocessor: C

In [30]:
from titanic_ml.common.experiments.v0_config import baseline_config

print("Baseline configuration:")
for key, value in baseline_config.items():
    print(f"{key}: {value}")

Baseline configuration:
logreg__baseline: {'name': 'baseline__raw__logreg', 'features': ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked'], 'feature_engineering': [], 'preprocessing': {'numeric_features': ['Age', 'SibSp', 'Parch', 'Fare'], 'onehot_features': ['Sex', 'Embarked'], 'ordinal_features': ['Pclass'], 'numeric_imputer': 'median', 'categorical_imputer': 'most_frequent', 'scaler': 'standard'}, 'model_name': 'logreg', 'model_params': {'max_iter': 1000, 'random_state': 42}, 'evaluation': {'method': 'cross_validation', 'cv': 5, 'scoring': ['accuracy', 'precision', 'recall', 'f1'], 'return_train_score': True, 'n_jobs': -1}, 'notes': 'Base logistic regression, using raw configuration. Baseline for comparison.', 'stage': 'baseline', 'feature_group': 'raw', 'group': 'baseline__raw'}
knn__baseline: {'name': 'baseline__raw__knn', 'features': ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked'], 'feature_engineering': [], 'preprocessing': {'numeric_features': ['Age',

In [53]:
from titanic_ml.common.experiments.v0_revised_runner import run_experiment_group_workflow as test_workflow

test_workflow_result = test_workflow(df=train_df, experiment_configs=baseline_config, target='Survived', save=True, verbose=True, debug=True)

Running experiment: baseline__raw__logreg
Running config: {'name': 'baseline__raw__logreg', 'features': ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked'], 'feature_engineering': [], 'preprocessing': {'numeric_features': ['Age', 'SibSp', 'Parch', 'Fare'], 'onehot_features': ['Sex', 'Embarked'], 'ordinal_features': ['Pclass'], 'numeric_imputer': 'median', 'categorical_imputer': 'most_frequent', 'scaler': 'standard'}, 'model_name': 'logreg', 'model_params': {'max_iter': 1000, 'random_state': 42}, 'evaluation': {'method': 'cross_validation', 'cv': 5, 'scoring': ['accuracy', 'precision', 'recall', 'f1'], 'return_train_score': True, 'n_jobs': -1}, 'notes': 'Base logistic regression, using raw configuration. Baseline for comparison.', 'stage': 'baseline', 'feature_group': 'raw', 'group': 'baseline__raw'}
Pipeline steps:
  preprocessor: ColumnTransformer
  model: LogisticRegression

Experiment 'baseline__raw__logreg' results:
  stage: baseline
  feature_group: raw
  model_name: lo

In [55]:
full_report = workflow_report(test_workflow_result, top_n=20)
print("Full workflow report:")
print()
print('Report')
print(full_report['report'])
print()
print('Leaderboard')
print(full_report['leaderboard'])
print()

from titanic_ml.common.experiments.v0_revised_save_load import load_results
# For combos:
all_results = load_results()
print("All results loaded:")
print(all_results)
# references = ['fe12__sex_pclass']
# for reference in references:
#     comparison = compare_experiment_groups(
#                 results_df=all_results,
#                 reference_group=reference,
#                 compare_groups=[exp_configs],
#             )
#     print(f"Comparison summary:")
#     print(comparison[["reference_group", "compare_group", "model_name", "test_accuracy_mean_delta", "test_f1_mean_delta"]].to_markdown())
#     print()

Full workflow report:

Report
### baseline__raw

_Description pending._

#### Conclusion

_Conclusion pending._

<details>
<summary>Details</summary>

#### Result

| model_name    | accuracy      | f1            |
|:--------------|:--------------|:--------------|
| logreg        | 0.786 ± 0.018 | 0.713 ± 0.026 |
| knn           | 0.809 ± 0.021 | 0.742 ± 0.026 |
| svc           | 0.827 ± 0.015 | 0.76 ± 0.026  |
| decision_tree | 0.803 ± 0.023 | 0.702 ± 0.055 |
| random_forest | 0.822 ± 0.02  | 0.744 ± 0.041 |
| extra_trees   | 0.804 ± 0.012 | 0.721 ± 0.025 |
| xgb           | 0.826 ± 0.025 | 0.758 ± 0.041 |

</details>

Leaderboard
| experiment                   | model_name    |   test_accuracy_mean |   test_f1_mean |
|:-----------------------------|:--------------|---------------------:|---------------:|
| baseline__raw__svc           | svc           |                0.827 |          0.76  |
| baseline__raw__xgb           | xgb           |                0.826 |          0.758 |
| bas

In [58]:
from titanic_ml.common.experiments.v0_revised_runner import run_experiments
from titanic_ml.common.experiments.v0_revised_save_load import save_results, save_configs, save_feature_effects



# print(f"Running {Name} experiments...")
exp_result = run_experiments(train_df, baseline_config, target=TARGET, verbose=True, debug=True)
result_df = save_results(exp_result)
save_configs(baseline_config)
# if Name != 'baseline__raw':
comparison = compare_experiment_groups(
    results_df=result_df,
    reference_group="baseline__raw",
    compare_groups=["baseline__raw"],
)
feature_effect = analyze_feature_effect(comparison)
save_feature_effects(feature_effect)

Running experiment: baseline__raw__logreg
Running config: {'name': 'baseline__raw__logreg', 'features': ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked'], 'feature_engineering': [], 'preprocessing': {'numeric_features': ['Age', 'SibSp', 'Parch', 'Fare'], 'onehot_features': ['Sex', 'Embarked'], 'ordinal_features': ['Pclass'], 'numeric_imputer': 'median', 'categorical_imputer': 'most_frequent', 'scaler': 'standard'}, 'model_name': 'logreg', 'model_params': {'max_iter': 1000, 'random_state': 42}, 'evaluation': {'method': 'cross_validation', 'cv': 5, 'scoring': ['accuracy', 'precision', 'recall', 'f1'], 'return_train_score': True, 'n_jobs': -1}, 'notes': 'Base logistic regression, using raw configuration. Baseline for comparison.', 'stage': 'baseline', 'feature_group': 'raw', 'group': 'baseline__raw'}
Pipeline steps:
  preprocessor: ColumnTransformer
  model: LogisticRegression

Experiment 'baseline__raw__logreg' results:
  stage: baseline
  feature_group: raw
  model_name: lo

{'baseline__raw': {'compare_group': 'baseline__raw',
  'metric': 'test_accuracy_mean',
  'mean_delta': np.float64(0.0),
  'max_delta': np.float64(0.0),
  'min_delta': np.float64(0.0),
  'positive_models': [],
  'neutral_models': ['logreg',
   'knn',
   'svc',
   'decision_tree',
   'random_forest',
   'extra_trees',
   'xgb'],
  'negative_models': [],
  'verdict': 'neutral',
  'recommended_for_all': False,
  'recommended_model_deltas': {},
  'recommended_model_tradeoffs': {},
  'notable_secondary_improvements': {},
  'discard': False}}

In [32]:
# import pprint
# Feature_effect = analyze_feature_effect(workflow['comparison'])
# pprint.pprint(Feature_effect)

In [33]:
# Reminder of how to run a single experiment if needed. 
# This is useful for when we want to generate reports or comparisons without rerunning all experiments.

# exp_config = [exp_config]
# exp_result = run_experiments(train_df, exp_config, target=TARGET,)
# exp_report = experiment_report(exp_result, exp_config, print_report=True)
# save_results(exp_result)
# save_configs(exp_config)

In [34]:
# Summary for baseline experiment to use in report, 
# since we can't compare it to itself.

# baseline_summary = baseline_summary_to_markdown(exp_result)
# print("Baseline summary:")
# print(baseline_summary)

In [35]:
# Reminder for how to load results and configs if needed. 
# This is useful for when we want to generate reports or comparisons without rerunning all experiments.

# all_results = load_results()
# print("Loaded results:")
# print(all_results)

# all_configs = load_configs()
# print("Loaded configs:")
# print(all_configs)

In [36]:
# print(workflow["all_results"])

In [37]:
# for model in MODEL_REGISTRY:
#     model_progression_df = model_progression(workflow["all_results"], model_name=model, metric="test_accuracy_mean")
#     print(f"Model progression for {model}:")
#     print(model_progression_df)
#     print()

Old Eda practice

In [38]:
# eda = run_eda(train_df, target=TARGET, display=True, head=3,random=4, tail=3)
# print(eda)

In [39]:
# print(eda)

In [40]:
# print(eda['dataFrame_health'].to_markdown())

In [41]:
# print(eda["dataFrame_summary"].to_markdown())

In [42]:
# for col in eda['categorical_summary']:
#     sample = sample_dataframe(eda['categorical_summary'][col], head=1, random=3, tail=1)
#     sample_df = pd.concat(
#         [sample[k] for k in ['head', 'random', 'tail']],
#         ignore_index=False
#     )
#     print(sample_df.to_markdown())
#     print()

In [43]:
# for col in eda['numerical_summary']:
#     print(f"Numerical column: {col}")
#     print(eda['numerical_summary'][col].to_markdown())
#     print()

In [44]:
# print("Correlation matrix:")
# print(eda["correlation_matrix"].to_markdown())

In [45]:
# print('Correlation with the target variable:')
# print(eda["target_correlation"].to_markdown())

In [46]:
# for col in eda["categorical_rare"]:
#     print(f"Categorical column with rare values: {col}")
#     print(eda["categorical_rare"][col])
#     print()

In [47]:
# # print(eda['sample'])
# sample_df = pd.concat(
#     [eda['sample'][k] for k in ['head', 'random', 'tail']],
#     ignore_index=True
# )
# print(sample_df.to_markdown(index=False))